In [13]:
import os
import random
import json
import re
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

Устройство: cuda


**Источник данных:** датасет `MakTek/Customer_support_faqs_dataset` с платформы Hugging Face.

**Характеристики:**
- Количество документов: 15
- Формат: пары «вопрос–ответ» (FAQ)
- Тематика: поддержка клиентов интернет-сервиса (регистрация, оплата, доставка, возврат, учётная запись)

**Примеры документов:**

| № | Вопрос | Ответ (фрагмент) |
|---|----------|------------------|
| 1 | How can I create an account? | To create an account, click on the 'Sign Up' button... |
| 2 | What payment methods do you accept? | We accept major credit cards, debit cards, and PayPal... |
| 3 | How can I track my order? | You can track your order by logging into your account... |

**Почему эта база подходит для retrieval / mini-RAG:**

1. **Чёткая семантика запросов.** Вопросы пользователей в FAQ имеют явную интенцию, что позволяет корректно оценивать релевантность извлечения.

2. **Однородная структура.** Все документы следуют единому шаблону «вопрос–ответ», что упрощает чанкинг и построение эмбеддингов.

3. **Практическая применимость.** Система может отвечать на типовые вопросы клиентов, извлекая соответствующие фрагменты из базы знаний — это классический сценарий RAG.

4. **Контролируемый масштаб.** 15 документов (после чанкинга ~30–50 фрагментов) достаточно для демонстрации pipeline, но не перегружает вычисления.

5. **Возможность оценки качества.** Для каждого запроса можно явно указать ожидаемый источник, что позволяет рассчитать hit@k, recall@k и другие метрики.

In [14]:
from datasets import load_dataset

dataset = load_dataset("MakTek/Customer_support_faqs_dataset", split="train")
dataset = dataset.select(range(15))

documents = []
for idx, row in enumerate(dataset, start=1):
    documents.append({
        "id": idx,
        "title": row["question"],
        "text": row["answer"]
    })

print(f"Загружено документов: {len(documents)}")
print(f"\nПримеры документов:")
for i in range(3):
    doc = documents[i]
    print(f"\n[{doc['id']}] {doc['title']}")
    print(f"    {doc['text'][:150]}...")

import os
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/knowledge_base.json", "w", encoding="utf-8") as f:
    import json
    json.dump(documents, f, ensure_ascii=False, indent=2)

Загружено документов: 15

Примеры документов:

[1] How can I create an account?
    To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration pr...

[2] What payment methods do you accept?
    We accept major credit cards, debit cards, and PayPal as payment methods for online orders....

[3] How can I track my order?
    You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for...


In [15]:
def simple_chunker(text, chunk_size=100, overlap=20):
    """Разбивает текст на чанки с перекрытием по словам"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
        if end >= len(words):
            break
    return chunks

CHUNK_SIZE = 100
OVERLAP = 20

import json
with open("artifacts/knowledge_base.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

chunks = []
for doc in documents:
    full_text = f"Q: {doc['title']} A: {doc['text']}"
    doc_chunks = simple_chunker(full_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "doc_id": doc["id"],
            "chunk_id": i,
            "text": chunk,
            "source": doc["title"]
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Параметры: chunk_size={CHUNK_SIZE}, overlap={OVERLAP}")

print(f"\nПример для документа [{documents[0]['id']}] {documents[0]['title']}:")
example_chunks = [c for c in chunks if c["doc_id"] == documents[0]["id"]]
for i, ch in enumerate(example_chunks):
    print(f"\nЧанк {i}: {ch['text']}")

import pandas as pd
chunks_df = pd.DataFrame(chunks)
chunks_df.to_csv("artifacts/chunks.csv", index=False, encoding="utf-8")

Всего чанков: 15
Параметры: chunk_size=100, overlap=20

Пример для документа [1] How can I create an account?:

Чанк 0: Q: How can I create an account? A: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.


In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

print("Загрузка модели эмбеддингов...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Модель загружена на устройство: {device}")

chunks_df = pd.read_csv("artifacts/chunks.csv")
chunk_texts = chunks_df["text"].tolist()

print(f"Генерация эмбеддингов для {len(chunk_texts)} чанков...")
embeddings = model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
print(f"Размерность эмбеддингов: {embeddings.shape}")

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(f"Индекс FAISS создан, добавлено векторов: {index.ntotal}")

faiss.write_index(index, "artifacts/faiss_index.bin")
np.save("artifacts/embeddings.npy", embeddings)
print("Индекс и эмбеддинги сохранены в artifacts/")

def search(query, top_k=3):
    query_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "score": float(distances[0][i]),
            "source": chunks_df.iloc[idx]["source"],
            "text": chunks_df.iloc[idx]["text"][:200] + "..."
        })
    return results

test_queries = [
    "How to sign up?",
    "What cards do you accept?",
    "Where is my order?"
]

print("\nПримеры поиска (top-3):")
for q in test_queries:
    print(f"\nЗапрос: '{q}'")
    results = search(q, top_k=3)
    for r in results:
        print(f"  [{r['rank']}] score={r['score']:.3f} | {r['source']}")
        print(f"       {r['text']}")

Загрузка модели эмбеддингов...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена на устройство: cuda
Генерация эмбеддингов для 15 чанков...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Размерность эмбеддингов: (15, 384)
Индекс FAISS создан, добавлено векторов: 15
Индекс и эмбеддинги сохранены в artifacts/

Примеры поиска (top-3):

Запрос: 'How to sign up?'
  [1] score=0.610 | How can I create an account?
       Q: How can I create an account? A: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process....
  [2] score=0.219 | How can I contact customer support?
       Q: How can I contact customer support? A: You can contact our customer support team by phone at [phone number] or by email at [email address]. Our team is available [working hours] to assist you with ...
  [3] score=0.196 | Can I order by phone?
       Q: Can I order by phone? A: Unfortunately, we do not accept orders over the phone. Please place your order through our website for a smooth and secure transaction....

Запрос: 'What cards do you accept?'
  [1] score=0.458 | What payment methods do you accept?

In [17]:
import pandas as pd
import numpy as np

control_queries = [
    {"query": "How do I create an account?", "expected_source": "How can I create an account?"},
    {"query": "What payment methods are supported?", "expected_source": "What payment methods do you accept?"},
    {"query": "How to track my shipment?", "expected_source": "How can I track my order?"},
    {"query": "Can I return a product?", "expected_source": "What is your return policy?"},
    {"query": "How to reset my password?", "expected_source": "I forgot my password, what should I do?"},
    {"query": "Do you ship internationally?", "expected_source": "Do you offer international shipping?"},
    {"query": "How to cancel an order?", "expected_source": "Can I cancel my order?"},
    {"query": "What is the delivery time?", "expected_source": "How long does shipping take?"},
    {"query": "How to contact support?", "expected_source": "How can I contact customer support?"},
    {"query": "Is my payment data safe?", "expected_source": "Are my personal and payment details secure?"}
]

K = 3  

results = []
for item in control_queries:
    query = item["query"]
    expected = item["expected_source"]
    query_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, K)    
    retrieved = [chunks_df.iloc[idx]["source"] for idx in indices[0]]
    hit = int(expected in retrieved)
    rank_first = None
    if expected in retrieved:
        rank_first = retrieved.index(expected) + 1
    
    results.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": "; ".join(retrieved),
        "hit_at_k": hit,
        "rank_of_first_relevant": rank_first
    })

df_eval = pd.DataFrame(results)
hit_at_k = df_eval["hit_at_k"].mean()
recall_at_k = hit_at_k
mrr = df_eval["rank_of_first_relevant"].dropna().apply(lambda x: 1/x).mean() if df_eval["rank_of_first_relevant"].notna().any() else 0.0

print(f"Оценка retrieval (k={K}):")
print(f"  Hit@{K}: {hit_at_k:.3f}")
print(f"  Recall@{K}: {recall_at_k:.3f}")
print(f"  MRR@{K}: {mrr:.3f}")

df_eval.to_csv("artifacts/retrieval_eval.csv", index=False, encoding="utf-8")

misses = df_eval[df_eval["hit_at_k"] == 0]
if len(misses) > 0:
    print(f"\nЗапросы с ошибкой retrieval ({len(misses)} из {len(df_eval)}):")
    for _, row in misses.iterrows():
        print(f"  Запрос: '{row['query']}'")
        print(f"  Ожидалось: '{row['expected_source']}'")
        print(f"  Найдено: {row['retrieved_sources']}")

Оценка retrieval (k=3):
  Hit@3: 0.900
  Recall@3: 0.900
  MRR@3: 1.000

Запросы с ошибкой retrieval (1 из 10):
  Запрос: 'How to reset my password?'
  Ожидалось: 'I forgot my password, what should I do?'
  Найдено: How can I create an account?; How can I contact customer support?; Can I cancel my order?


In [18]:
import pandas as pd

control_queries = [
    {"query": "How do I create an account?", "expected_source": "How can I create an account?"},
    {"query": "What payment methods are supported?", "expected_source": "What payment methods do you accept?"},
    {"query": "How to track my shipment?", "expected_source": "How can I track my order?"},
    {"query": "Can I return a product?", "expected_source": "What is your return policy?"},
    {"query": "How to reset my password?", "expected_source": "I forgot my password, what should I do?"},
    {"query": "Do you ship internationally?", "expected_source": "Do you offer international shipping?"},
    {"query": "How to cancel an order?", "expected_source": "Can I cancel my order?"},
    {"query": "What is the delivery time?", "expected_source": "How long does shipping take?"},
    {"query": "How to contact support?", "expected_source": "How can I contact customer support?"},
    {"query": "Is my payment data safe?", "expected_source": "Are my personal and payment details secure?"}
]

def evaluate_top_k(k):
    results = []
    for item in control_queries:
        query = item["query"]
        expected = item["expected_source"]
        query_emb = model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        distances, indices = index.search(query_emb, k)
        retrieved = [chunks_df.iloc[idx]["source"] for idx in indices[0]]
        hit = int(expected in retrieved)
        results.append({"query": query, "expected": expected, "hit": hit, "retrieved": retrieved})
    df = pd.DataFrame(results)
    return df["hit"].mean(), df

hit_3, df_3 = evaluate_top_k(3)
hit_5, df_5 = evaluate_top_k(5)

print("Сравнение параметров retrieval:")
print(f"  top_k=3: Hit@3 = {hit_3:.3f}")
print(f"  top_k=5: Hit@5 = {hit_5:.3f}")
print(f"  Прирост: {(hit_5 - hit_3) * 100:.1f}%")

improved = []
for i, item in enumerate(control_queries):
    if df_3.iloc[i]["hit"] == 0 and df_5.iloc[i]["hit"] == 1:
        improved.append({
            "query": item["query"],
            "expected": item["expected_source"],
            "retrieved_at_5": "; ".join(df_5.iloc[i]["retrieved"])
        })

if improved:
    print(f"\nЗапросы, которые стали релевантными при top_k=5 ({len(improved)} шт.):")
    for imp in improved:
        print(f"  '{imp['query']}' -> ожидалось: '{imp['expected']}'")
else:
    print("\nУвеличение top_k не улучшило результаты на данном наборе запросов.")

print(f"\nВывод: {'увеличение top_k улучшает recall, но может добавить шум' if hit_5 > hit_3 else 'top_k=3 достаточно для данной базы'}")

Сравнение параметров retrieval:
  top_k=3: Hit@3 = 0.900
  top_k=5: Hit@5 = 0.900
  Прирост: 0.0%

Увеличение top_k не улучшило результаты на данном наборе запросов.

Вывод: top_k=3 достаточно для данной базы


In [19]:
import json
import os
import pandas as pd
import numpy as np
import faiss

with open("artifacts/knowledge_base.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Документов до обновления: {len(documents)}")

new_docs = [
    {
        "id": 16,
        "title": "I forgot my password, what should I do?",
        "text": "If you forgot your password, click on the 'Forgot Password' link on the login page. Enter your email address and we will send you a password reset link. The link is valid for 24 hours. If you don't receive the email, check your spam folder or contact support."
    },
    {
        "id": 17,
        "title": "How do I change my email address?",
        "text": "To change your email address, go to Account Settings > Profile > Email. Enter your new email and confirm it via the verification link we send. Your old email will remain active until you verify the new one."
    },
    {
        "id": 18,
        "title": "What is two-factor authentication?",
        "text": "Two-factor authentication (2FA) adds an extra layer of security to your account. After entering your password, you will need to confirm your identity via a code sent to your phone or authenticator app. Enable 2FA in Account Settings > Security."
    }
]

documents.extend(new_docs)
print(f"Документов после обновления: {len(documents)}")

def simple_chunker(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
        if end >= len(words):
            break
    return chunks

CHUNK_SIZE = 100
OVERLAP = 20

new_chunks = []
for doc in documents:
    full_text = f"Q: {doc['title']} A: {doc['text']}"
    doc_chunks = simple_chunker(full_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        new_chunks.append({
            "doc_id": doc["id"],
            "chunk_id": i,
            "text": chunk,
            "source": doc["title"]
        })

print(f"Чанков после обновления: {len(new_chunks)}")

print("Генерация новых эмбеддингов...")
new_chunk_texts = [c["text"] for c in new_chunks]
new_embeddings = model.encode(new_chunk_texts, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(new_embeddings)

dimension = new_embeddings.shape[1]
new_index = faiss.IndexFlatIP(dimension)
new_index.add(new_embeddings)
print(f"Новый индекс создан: {new_index.ntotal} векторов")

test_queries = [
    "How to reset my password?",
    "How to enable 2FA?",
    "Change my email"
]

comparison_results = []
for query in test_queries:
    query_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    
    # Поиск в старом индексе
    _, old_indices = index.search(query_emb, 3)
    old_sources = [chunks_df.iloc[idx]["source"] for idx in old_indices[0]]
    
    # Поиск в новом индексе
    _, new_indices = new_index.search(query_emb, 3)
    new_chunks_df = pd.DataFrame(new_chunks)
    new_sources = [new_chunks_df.iloc[idx]["source"] for idx in new_indices[0]]
    
    changed = old_sources != new_sources
    comparison_results.append({
        "query": query,
        "before_retrieved_sources": "; ".join(old_sources),
        "after_retrieved_sources": "; ".join(new_sources),
        "changed": changed
    })
    
    print(f"\nЗапрос: '{query}'")
    print(f"  До: {old_sources}")
    print(f"  После: {new_sources}")
    print(f"  Изменилось: {changed}")

df_comparison = pd.DataFrame(comparison_results)
df_comparison.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding="utf-8")

new_chunks_df = pd.DataFrame(new_chunks)
new_chunks_df.to_csv("artifacts/chunks_updated.csv", index=False, encoding="utf-8")
faiss.write_index(new_index, "artifacts/faiss_index_updated.bin")
np.save("artifacts/embeddings_updated.npy", new_embeddings)

Документов до обновления: 15
Документов после обновления: 18
Чанков после обновления: 18
Генерация новых эмбеддингов...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Новый индекс создан: 18 векторов

Запрос: 'How to reset my password?'
  До: ['How can I create an account?', 'How can I contact customer support?', 'Can I cancel my order?']
  После: ['I forgot my password, what should I do?', 'What is two-factor authentication?', 'How do I change my email address?']
  Изменилось: True

Запрос: 'How to enable 2FA?'
  До: ['How can I create an account?', 'Are my personal and payment details secure?', 'What payment methods do you accept?']
  После: ['What is two-factor authentication?', 'How can I create an account?', 'I forgot my password, what should I do?']
  Изменилось: True

Запрос: 'Change my email'
  До: ['Can I change my shipping address after placing an order?', 'How can I contact customer support?', 'Can I cancel my order?']
  После: ['How do I change my email address?', 'I forgot my password, what should I do?', 'Can I change my shipping address after placing an order?']
  Изменилось: True


In [20]:
import pandas as pd

class MiniRAG:
    """Простой RAG-конвейер для учебной демонстрации"""
    
    def __init__(self, index, chunks_df, model, top_k=3):
        self.index = index
        self.chunks_df = chunks_df
        self.model = model
        self.top_k = top_k
    
    def retrieve(self, query):
        """Извлекает top-k релевантных чанков"""
        query_emb = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        distances, indices = self.index.search(query_emb, self.top_k)
        results = []
        for i, idx in enumerate(indices[0]):
            results.append({
                "rank": i + 1,
                "score": float(distances[0][i]),
                "source": self.chunks_df.iloc[idx]["source"],
                "text": self.chunks_df.iloc[idx]["text"]
            })
        return results
    
    def build_context(self, retrieved_chunks):
        """Формирует контекст из найденных фрагментов"""
        context_parts = []
        for chunk in retrieved_chunks:
            context_parts.append(f"[{chunk['source']}]: {chunk['text']}")
        return "\n\n".join(context_parts)
    
    def generate_answer(self, query, context):
        """Генерирует ответ на основе контекста (шаблонный подход)"""
        query_words = set(query.lower().split())
        sentences = context.replace("\n", " ").split(". ")
        relevant_sentences = []
        for sent in sentences:
            sent_words = set(sent.lower().split())
            if len(query_words & sent_words) >= 2:  # минимум 2 общих слова
                relevant_sentences.append(sent.strip())
        
        if relevant_sentences:
            answer = ". ".join(relevant_sentences[:2]) + "."
        else:
            answer = context.split("\n\n")[0].split("]: ", 1)[-1] if context else "Информация не найдена."
        
        return answer
    
    def answer(self, query):
        """Полный цикл: запрос -> retrieval -> контекст -> ответ"""
        retrieved = self.retrieve(query)
        context = self.build_context(retrieved)
        answer = self.generate_answer(query, context)
        sources = [r["source"] for r in retrieved]
        return {
            "question": query,
            "answer": answer,
            "retrieved_sources": sources,
            "context": context
        }

new_chunks_df = pd.read_csv("artifacts/chunks_updated.csv")
rag = MiniRAG(new_index, new_chunks_df, model, top_k=3)

test_questions = [
    "How do I reset my password?",
    "What is two-factor authentication?",
    "How to change my email?",
    "What payment methods do you accept?",
    "Can I track my order?"
]

print("Примеры работы mini-RAG:\n")
for q in test_questions:
    result = rag.answer(q)
    print(f"Вопрос: {result['question']}")
    print(f"Ответ: {result['answer']}")
    print(f"Источники: {', '.join(result['retrieved_sources'])}")
    print("-" * 80)

rag_examples = []
for q in test_questions:
    result = rag.answer(q)
    rag_examples.append({
        "question": result["question"],
        "answer": result["answer"],
        "retrieved_sources": "; ".join(result["retrieved_sources"])
    })

df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv("artifacts/rag_examples.csv", index=False, encoding="utf-8")

Примеры работы mini-RAG:

Вопрос: How do I reset my password?
Ответ: [I forgot my password, what should I do?]: Q: I forgot my password, what should I do? A: If you forgot your password, click on the 'Forgot Password' link on the login page. [How do I change my email address?]: Q: How do I change my email address? A: To change your email address, go to Account Settings > Profile > Email.
Источники: I forgot my password, what should I do?, What is two-factor authentication?, How do I change my email address?
--------------------------------------------------------------------------------
Вопрос: What is two-factor authentication?
Ответ: [What is two-factor authentication?]: Q: What is two-factor authentication? A: Two-factor authentication (2FA) adds an extra layer of security to your account.
Источники: What is two-factor authentication?, Are my personal and payment details secure?, How can I create an account?
---------------------------------------------------------------------------

### 2.3.9. Краткий анализ ошибок

**Удачные кейсы:**

| Вопрос | Результат | Причина успеха |
|--------|-----------|----------------|
| "What is two-factor authentication?" | Точный ответ из нужного источника | Прямое совпадение терминов в запросе и документе |
| "What payment methods do you accept?" | Корректный ответ с перечислением способов оплаты | Высокая семантическая близость запроса к заголовку документа |

**Пограничные случаи:**

1. **Запрос:** "How do I reset my password?"  
   **Ответ:** Содержит информацию о сбросе пароля, но также включает фрагмент про смену email.  
   **Причина:** Эвристика генерации ответа выбирает предложения с ≥2 общими словами с запросом. Слова "email", "address" попали в выборку из-за перекрытия с "password reset" контекстом.  
   **Улучшение:** Использовать более строгий фильтр по семантической релевантности или добавить reranking.

2. **Запрос:** "How to change my email?"  
   **Ответ:** Включает фрагмент про изменение адреса доставки (shipping address).  
   **Причина:** Лексическое перекрытие: слова "change", "address" встречаются в разных контекстах.  
   **Улучшение:** Добавить пост-обработку для исключения нерелевантных сущностей (shipping ≠ email).

3. **Запрос:** "Can I track my order?"  
   **Ответ:** Корректный, но второй источник ("Can I order by phone?") не релевантен.  
   **Причина:** Векторная близость из-за общих слов "order", "phone" в контексте покупок.  
   **Улучшение:** Увеличить weight заголовка при построении эмбеддингов или использовать гибридный поиск.

**Общие ограничения текущего mini-RAG:**

- Шаблонная генерация ответа не учитывает структуру диалога.
- Нет обработки отрицаний и сложных логических условий.
- Контекст собирается из top-k фрагментов без учёта их взаимной согласованности.

**Вывод:** Для учебной демонстрации качество приемлемо. Для продакшена потребуется: (1) более сложная генерация через LLM, (2) reranking результатов retrieval, (3) валидация ответа на соответствие источникам.